# Confidence-Driven Procurement Agent with ShopGraph

When a procurement agent auto-fills a purchase order, a wrong price is worse
than a missing price. This cookbook builds a LangChain agent that uses per-field
confidence scores to decide: act autonomously on high-confidence fields, route
low-confidence fields to human review.

**The core problem:** Extraction APIs return data. They don't tell you how much
to trust each field. A price pulled from a Schema.org tag is more reliable than
a price inferred from surrounding text. Without per-field confidence, the agent
either trusts everything (writes bad data into POs) or trusts nothing (you might
as well not have an agent).

**What ShopGraph adds:** Per-field confidence scores (0.0 to 1.0) on every
extracted field, plus server-side threshold filtering that scrubs uncertain
fields from the response before the agent can see them.

In [ ]:
%pip install requests langchain langchain-openai -q


In [ ]:
import os
import json
import requests

SHOPGRAPH_API_KEY = os.environ.get("SHOPGRAPH_API_KEY", "your-api-key")
SHOPGRAPH_API_URL = "https://shopgraph.dev/api/enrich"

# Confidence baselines by extraction method:
#   Schema.org / JSON-LD (tier 1): ~0.90-0.95
#   LLM extraction (tier 2):      ~0.65-0.80
#   Headless browser (tier 3):    ~0.50-0.70
#
# Scores reflect cross-tier agreement, not just one method's output
# probability. When two tiers agree on a price and one dissents,
# the confidence reflects that signal.


## Two extraction modes

Procurement agents make two kinds of calls:

1. **Research mode.** A human reviews the output. The agent wants every field
   including low-confidence ones, so the human can judge what to trust.

2. **Autofill mode.** The agent writes data directly into a PO, inventory
   system, or RFQ response. No human reviews it. The agent wants only fields
   confident enough to act on. Anything below threshold should be absent,
   not flagged.

The difference matters because of context window contamination: if the agent
sees a low-confidence price in research mode, it may reference that price
later during autofill even though it shouldn't. Server-side filtering
(`strict_confidence_threshold`) removes the temptation entirely. The field
never enters the agent's context.

In [ ]:
from langchain_core.tools import tool

@tool
def extract_product(url: str) -> str:
    """Extract product data with per-field confidence scores from a commerce URL.

    Use for research calls where a human will review the output. Returns all
    fields regardless of confidence, so the human can judge what to trust.

    Each field includes a confidence score (0.0 to 1.0):
      0.90+     extracted from structured markup, high reliability
      0.70-0.89 extracted via LLM, moderate reliability
      below 0.70 inferred or partially matched, verify before relying on

    Args:
        url: Product page URL.

    Returns:
        JSON string with product data and per-field confidence scores.
    """
    response = requests.post(
        SHOPGRAPH_API_URL,
        headers={"Authorization": f"Bearer {SHOPGRAPH_API_KEY}"},
        json={"url": url},
        timeout=30,
    )
    if not response.ok:
        return json.dumps({"error": True, "status": response.status_code, "url": url})
    return json.dumps(response.json(), indent=2)


In [ ]:
@tool
def extract_product_for_autofill(url: str, threshold: float = 0.9) -> str:
    """Extract product data, returning only fields above the confidence threshold.

    Use when writing extracted values directly into a purchase order, inventory
    system, or RFQ response without human review. Fields below threshold are
    scrubbed server-side and will not appear in the response.

    A missing field means 'not confident enough to act on.' The agent should
    not attempt to backfill missing fields from other sources or from earlier
    in the conversation without flagging for human review.

    Args:
        url: Product page URL.
        threshold: Minimum confidence (0.0 to 1.0). Default 0.9 is appropriate
            for most autofill use cases. Use 0.95 for contract pricing or
            regulated goods. Use 0.8 only when the downstream system has its
            own validation step.

    Returns:
        JSON string with only high-confidence fields. Missing fields were
        below threshold and intentionally excluded.
    """
    response = requests.post(
        SHOPGRAPH_API_URL,
        headers={"Authorization": f"Bearer {SHOPGRAPH_API_KEY}"},
        json={"url": url, "strict_confidence_threshold": threshold},
        timeout=30,
    )
    if not response.ok:
        return json.dumps({"error": True, "status": response.status_code, "url": url})
    return json.dumps(response.json(), indent=2)


## How confidence drives the autonomy decision

The agent doesn't decide trust levels. The API does.

When `extract_product` returns all fields with confidence scores, the
agent reports them to the human. The human decides.

When `extract_product_for_autofill` is called with `threshold=0.9`,
the API scrubs every field below 0.9 before the response leaves the
server. The agent literally cannot see uncertain data. If it needs a
field that was scrubbed, it must stop and ask the human.

This is the difference between "the agent is trusted to ignore bad data"
and "the agent cannot see the bad data." For autonomous writes into
procurement systems, the second is the only defensible posture.

In [ ]:
# Research mode: extract all fields with confidence scores
MOGLIX_URL = "https://www.moglix.com/bosch-1-2-inch-impact-wrench-gds-18-v-ec-250/mp/msne9bg5j9egz8"

result = json.loads(extract_product.invoke(MOGLIX_URL))

if "error" in result:
    print(f"Extraction failed: {result}")
else:
    product = result.get("product", result)
    shopgraph_meta = product.get("_shopgraph", {})
    field_confidence = shopgraph_meta.get("field_confidence", {})

    print("Extracted fields with confidence:\n")
    for field, conf in sorted(field_confidence.items(), key=lambda x: -x[1]):
        status = "OK" if conf >= 0.85 else "VERIFY" if conf >= 0.50 else "SKIP"
        print(f"  {field:20s}  confidence: {conf:.2f}  [{status}]")


## LangChain agent with two-tool autonomy routing

The system prompt teaches the agent the decision rule:

- **Researching a product?** Use `extract_product`. Show all fields with
  confidence. Flag anything below 0.85.
- **Auto-filling a PO or writing into a system?** Use
  `extract_product_for_autofill`. Default threshold 0.9. If a required
  field is missing from the response, stop and ask the human.

The agent never mixes modes in a single operation. Research first, confirm
the fields, then autofill.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o", temperature=0)

tools = [extract_product, extract_product_for_autofill]

system_prompt = """You are a procurement assistant with two extraction modes.

Choose based on what the task requires:

RESEARCH MODE (extract_product):
Call this when the human is researching a product, comparing suppliers, or
asking you to summarize what's available at a URL. Return all fields with
their confidence scores. Flag any field below 0.85 as "verify before
relying on." The human makes the trust decision.

AUTOFILL MODE (extract_product_for_autofill):
Call this when the human asks you to fill a purchase order, update an
inventory record, or write extracted values into any downstream system
without human review. Default threshold: 0.9. Use 0.95 for contract
pricing or regulated goods.

If a required field is missing from the autofill response, do NOT invent
it, guess it, or pull it from earlier in the conversation. Stop and tell
the human: "This field was below the confidence threshold and needs manual
verification."

Never mix modes in a single operation. If the task spans both research
and autofill, do the research call first, confirm the fields with the
human, then make the autofill call."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


## Demo: research then autofill

Watch the tool calls. The agent should:

1. Call `extract_product` first (research mode, all fields visible)
2. Report confidence scores to you
3. Call `extract_product_for_autofill` with threshold 0.9 (autofill mode)
4. Report which fields survived the threshold and which were scrubbed

If a field the PO needs was scrubbed, the agent should stop and ask
rather than invent a value.

In [ ]:
result = executor.invoke({
    "input": (
        f"Research this product: {MOGLIX_URL}. "
        f"Show me all extracted fields with their confidence scores. "
        f"Then, assuming the research looks good, prepare a PO line item "
        f"for 3 units using only fields with confidence above 0.9."
    )
})

print("\n" + "=" * 60)
print("AGENT OUTPUT:")
print("=" * 60)
print(result["output"])


## Why server-side filtering matters

Client-side filtering (checking confidence after the response arrives)
still lets the agent see low-confidence data. In a long context window,
the agent may reference a scrubbed price field from earlier in the
conversation because it was visible during the research step.

Server-side filtering via `strict_confidence_threshold` removes the field
from the API response entirely. The agent cannot reference what it never
received.

For autonomous writes into procurement systems, inventory databases, or
any system where a wrong value has downstream consequences: use
server-side filtering. The pattern is `extract_product_for_autofill`
with a threshold matched to the stakes of the write.

## Reference

**Pricing:**
Playground: 50 extractions/month, no signup required.
Starter: $99/month for 10K calls with API key.

**Confidence baselines by extraction method:**
| Method | Typical range | When used |
|---|---|---|
| Schema.org / JSON-LD (tier 1) | 0.90 to 0.95 | Site has structured markup |
| LLM extraction (tier 2) | 0.65 to 0.80 | No structured markup, page has readable product content |
| Headless browser (tier 3) | 0.50 to 0.70 | Content requires JavaScript rendering |

Scores reflect cross-tier agreement. When multiple tiers extract the same
field, agreement raises confidence; disagreement lowers it.

**Additional API options (not used in this cookbook):**
- AgentReady scoring: `?include_score=true` returns a 0-100 readiness score
- UCP output: `?format=ucp` for Universal Commerce Protocol schema
- Leaderboard: [shopgraph.dev/leaderboard](https://shopgraph.dev/leaderboard) shows which sites extract successfully

**Related:**
- [Vercel AI SDK example](https://github.com/vercel/ai) for client-side
  confidence rendering in a Next.js UI (complementary pattern: this cookbook
  does server-side filtering, that example does client-side rendering)